Primer código donde se realiza la comprobación del dataset, en cuestion de dimensiones y carga de los peores casos

In [1]:
#from google.colab import drive
import os

# 📌 Montar Google Drive
#drive.mount('/content/drive')

# 📌 Ruta del dataset
dataset_path = "NEH_UT_2021RetinalOCTDataset"

# 📌 Verificar archivos en la carpeta
print("📂 Archivos disponibles en el dataset:")
print(os.listdir(dataset_path))

📂 Archivos disponibles en el dataset:
['CNV', 'DRUSEN', 'NORMAL']


In [5]:
import cv2
import os
import random
from collections import Counter

# Ruta del dataset
DATASET_PATH = "NEH_UT_2021RetinalOCTDataset"

# Muestra a verificar
SAMPLE_SIZE = 50
SEED = 42
random.seed(SEED)

# Extensiones válidas
VALID_EXTS = (".jpg", ".jpeg", ".png")

# Lista de todas las imágenes disponibles en el dataset
image_paths = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(VALID_EXTS):
            image_paths.append(os.path.join(root, file))

print(f"Total de imágenes encontradas: {len(image_paths)}")

# Selección aleatoria de imágenes
sample_images = random.sample(image_paths, min(SAMPLE_SIZE, len(image_paths)))

# Conteo de tamaños encontrados
image_sizes = Counter()
unreadable = []

# Verificamos los tamaños de las imágenes seleccionadas aleatoriamente
for img_path in sample_images:
    img = cv2.imread(img_path)
    if img is not None:
        image_sizes[img.shape] += 1
    else:
        unreadable.append(img_path)

# Imprimimos resultados
print("\nTamaños de imagen en la muestra:")
for shape, count in image_sizes.items():
    print(f"  {shape}: {count} imagen(es)")

if unreadable:
    print(f"\nImágenes no leídas: {len(unreadable)}")

Total de imágenes encontradas: 3560

Tamaños de imagen en la muestra:
  (496, 768, 3): 50 imagen(es)


In [6]:
import os
import pandas as pd

CSV_PATH = "data_information.csv"

df = pd.read_csv(CSV_PATH)

print(f"📌 Total de imágenes en el CSV: {len(df)}")
print("\nColumnas disponibles:")
print(df.columns.tolist())

# Renombrar columnas para trabajar más cómodo
df = df.rename(columns={
    "Patient ID": "patient_id",
    "Class": "patient_class",
    "Eye": "eye",
    "B-scan": "bscan",
    "Label": "label",
    "Directory": "relative_path"
})

# Ruta completa de imagen
df["image_path"] = df["relative_path"].apply(lambda x: os.path.join(DATASET_PATH, x))

# Verificar existencia física de archivos
df["exists"] = df["image_path"].apply(os.path.exists)

print("\nImágenes existentes:", df["exists"].sum())
print("Imágenes faltantes:", (~df["exists"]).sum())

if (~df["exists"]).sum() > 0:
    display(df.loc[~df["exists"], ["relative_path", "image_path"]].head())

# Filtrar solo imágenes existentes
df = df[df["exists"]].reset_index(drop=True)

# Grupo real para evitar fuga:
# Patient ID solo no basta, porque se repite entre clases.
df["group_id"] = df["patient_class"].astype(str) + "_" + df["patient_id"].astype(str)

# Etiquetas normalizadas
df["label"] = df["label"].astype(str).str.upper().str.strip()

# Mapeo de clases
class_names = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["label_id"] = df["label"].map(class_to_idx)

print("\nDistribución de imágenes por clase (label):")
print(df["label"].value_counts())

print("\nNúmero de grupos únicos:")
print(df["group_id"].nunique())

print("\nClases encontradas:")
print(class_to_idx)

# Verificación de coherencia grupo-etiqueta
group_label_counts = df.groupby("group_id")["label"].nunique()
bad_groups = group_label_counts[group_label_counts > 1]

print(f"\nGrupos con más de una etiqueta: {len(bad_groups)}")
if len(bad_groups) > 0:
    display(df[df["group_id"].isin(bad_groups.index)].sort_values("group_id").head(20))

# Variables listas para StratifiedGroupKFold
X = df["image_path"].values
y = df["label_id"].values
groups = df["group_id"].values

display(df.head())

📌 Total de imágenes en el CSV: 16822

Columnas disponibles:
['Patient ID', 'Class', 'Eye', 'B-scan', 'Label', 'Directory']

Imágenes existentes: 16822
Imágenes faltantes: 0

Distribución de imágenes por clase (label):
label
NORMAL    8584
DRUSEN    4998
CNV       3240
Name: count, dtype: int64

Número de grupos únicos:
441

Clases encontradas:
{'CNV': 0, 'DRUSEN': 1, 'NORMAL': 2}

Grupos con más de una etiqueta: 268


,patient_id,patient_class,eye,bscan,label,relative_path,image_path,exists,group_id,label_id
0,1,CNV,OS,0,DRUSEN,CNV/1/000_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/000_Drusen.jpg,True,CNV_1,1
29,1,CNV,OS,29,NORMAL,CNV/1/029_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/029_Normal.jpg,True,CNV_1,2
28,1,CNV,OS,28,NORMAL,CNV/1/028_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/028_Normal.jpg,True,CNV_1,2
27,1,CNV,OS,27,NORMAL,CNV/1/027_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/027_Normal.jpg,True,CNV_1,2
26,1,CNV,OS,26,NORMAL,CNV/1/026_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/026_Normal.jpg,True,CNV_1,2
25,1,CNV,OS,25,NORMAL,CNV/1/025_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/025_Normal.jpg,True,CNV_1,2
24,1,CNV,OS,24,NORMAL,CNV/1/024_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/024_Normal.jpg,True,CNV_1,2
23,1,CNV,OS,23,NORMAL,CNV/1/023_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/023_Normal.jpg,True,CNV_1,2
22,1,CNV,OS,22,DRUSEN,CNV/1/022_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/022_Drusen.jpg,True,CNV_1,1
21,1,CNV,OS,21,DRUSEN,CNV/1/021_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/021_Drusen.jpg,True,CNV_1,1


,patient_id,patient_class,eye,bscan,label,relative_path,image_path,exists,group_id,label_id
0,1,CNV,OS,0,DRUSEN,CNV/1/000_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/000_Drusen.jpg,True,CNV_1,1
1,1,CNV,OS,1,DRUSEN,CNV/1/001_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/001_Drusen.jpg,True,CNV_1,1
2,1,CNV,OS,2,DRUSEN,CNV/1/002_Drusen.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/002_Drusen.jpg,True,CNV_1,1
3,1,CNV,OS,3,NORMAL,CNV/1/003_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/003_Normal.jpg,True,CNV_1,2
4,1,CNV,OS,4,NORMAL,CNV/1/004_Normal.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/004_Normal.jpg,True,CNV_1,2


In [7]:
# Filtrado de "peores casos":
# conservar solo imágenes cuya etiqueta individual coincide con la clase declarada del caso en la metadata

df_worst_cases = df[df["patient_class"] == df["label"]].copy().reset_index(drop=True)

print(f"📌 Total de imágenes después del filtro (peores casos): {len(df_worst_cases)}")

print("\nDistribución por clase después del filtro:")
print(df_worst_cases["label"].value_counts())

print("\nNúmero de grupos únicos después del filtro:")
print(df_worst_cases["group_id"].nunique())

# Variables base para StratifiedGroupKFold
X = df_worst_cases["image_path"].values
y = df_worst_cases["label_id"].values
groups = df_worst_cases["group_id"].values

display(df_worst_cases.head())

📌 Total de imágenes después del filtro (peores casos): 12649

Distribución por clase después del filtro:
label
NORMAL    5667
DRUSEN    3742
CNV       3240
Name: count, dtype: int64

Número de grupos únicos después del filtro:
441


,patient_id,patient_class,eye,bscan,label,relative_path,image_path,exists,group_id,label_id
0,1,CNV,OS,11,CNV,CNV/1/011_CNV.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/011_CNV.jpg,True,CNV_1,0
1,1,CNV,OS,12,CNV,CNV/1/012_CNV.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/012_CNV.jpg,True,CNV_1,0
2,1,CNV,OS,13,CNV,CNV/1/013_CNV.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/013_CNV.jpg,True,CNV_1,0
3,1,CNV,OS,14,CNV,CNV/1/014_CNV.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/014_CNV.jpg,True,CNV_1,0
4,1,CNV,OS,15,CNV,CNV/1/015_CNV.jpg,NEH_UT_2021RetinalOCTDataset\CNV/1/015_CNV.jpg,True,CNV_1,0


In [ ]:
import os

# Verificar existencia de archivos en el dataset filtrado
df_worst_cases["exists"] = df_worst_cases["image_path"].apply(os.path.exists)

print("📌 Verificación de rutas:")
print("Imágenes existentes:", df_worst_cases["exists"].sum())
print("Imágenes faltantes:", (~df_worst_cases["exists"]).sum())

# Mostrar ejemplos de rutas que fallan
if (~df_worst_cases["exists"]).sum() > 0:
    print("\n⚠️ Ejemplos de imágenes faltantes:")
    display(df_worst_cases.loc[~df_worst_cases["exists"], ["relative_path", "image_path"]].head())

# Filtrar solo imágenes válidas
df_worst_cases = df_worst_cases[df_worst_cases["exists"]].reset_index(drop=True)

print("\n📌 Dataset final después de limpiar rutas:")
print("Total de imágenes:", len(df_worst_cases))

print("\nDistribución por clase:")
print(df_worst_cases["label"].value_counts())

print("\nNúmero de grupos únicos:")
print(df_worst_cases["group_id"].nunique())

📌 Verificación de rutas:
Imágenes existentes: 12649
Imágenes faltantes: 0

📌 Dataset final después de limpiar rutas:
Total de imágenes: 12649

Distribución por clase:
label
NORMAL    5667
DRUSEN    3742
CNV       3240
Name: count, dtype: int64

Número de grupos únicos:
441


In [ ]:
# Variables finales para StratifiedGroupKFold
X = df_worst_cases["image_path"].values
y = df_worst_cases["label_id"].values
groups = df_worst_cases["group_id"].values

print("Shapes:")
print("X:", len(X))
print("y:", len(y))
print("groups:", len(groups))

Shapes:
X: 12649
y: 12649
groups: 12649


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

print("=" * 80)
print("VALIDACIÓN PREVIA A STRATIFIEDGROUPKFOLD")
print("=" * 80)

# -------------------------------------------------------------------
# 1) Validación básica de tamaños
# -------------------------------------------------------------------
print("\n[1] Tamaños de X, y, groups")
print(f"len(X):      {len(X)}")
print(f"len(y):      {len(y)}")
print(f"len(groups): {len(groups)}")

assert len(X) == len(y) == len(groups), "❌ X, y y groups no tienen la misma longitud."
print("✅ Longitudes consistentes.")

# -------------------------------------------------------------------
# 2) Validación del dataframe base
# -------------------------------------------------------------------
print("\n[2] Tamaño del dataframe final")
print(f"Filas en df_worst_cases: {len(df_worst_cases)}")
assert len(df_worst_cases) == len(X), "❌ df_worst_cases y X no coinciden en longitud."
print("✅ df_worst_cases coincide con X.")

# -------------------------------------------------------------------
# 3) Distribución de clases
# -------------------------------------------------------------------
print("\n[3] Distribución de clases (imágenes)")
class_counts = df_worst_cases["label"].value_counts().sort_index()
class_percent = df_worst_cases["label"].value_counts(normalize=True).sort_index() * 100

dist_df = pd.DataFrame({
    "count": class_counts,
    "percent": class_percent.round(2)
})
display(dist_df)

# También con label_id
print("\nDistribución por label_id:")
print(df_worst_cases["label_id"].value_counts().sort_index())

# -------------------------------------------------------------------
# 4) Grupos únicos
# -------------------------------------------------------------------
print("\n[4] Información de grupos")
n_unique_groups = df_worst_cases["group_id"].nunique()
print(f"Número de grupos únicos: {n_unique_groups}")

group_sizes = df_worst_cases.groupby("group_id").size()

print("\nResumen de imágenes por grupo:")
print(group_sizes.describe())

print("\nTop 10 grupos con más imágenes:")
display(group_sizes.sort_values(ascending=False).head(10).to_frame("n_images"))

print("\nTop 10 grupos con menos imágenes:")
display(group_sizes.sort_values(ascending=True).head(10).to_frame("n_images"))

# -------------------------------------------------------------------
# 5) Verificar si un grupo tiene más de una etiqueta
#    Idealmente NO debe ocurrir si group_id está bien definido
# -------------------------------------------------------------------
print("\n[5] Coherencia grupo-etiqueta")
group_label_nunique = df_worst_cases.groupby("group_id")["label"].nunique()
bad_groups = group_label_nunique[group_label_nunique > 1]

print(f"Grupos con más de una etiqueta: {len(bad_groups)}")

if len(bad_groups) > 0:
    print("⚠️ Se encontraron grupos con múltiples etiquetas. Revisar estos casos:")
    display(
        df_worst_cases[df_worst_cases["group_id"].isin(bad_groups.index)]
        .sort_values(["group_id", "label"])
        .head(30)
    )
else:
    print("✅ Cada grupo pertenece a una sola etiqueta.")

# -------------------------------------------------------------------
# 6) Número de grupos por clase
#    Esto es clave para saber si 5 folds es viable
# -------------------------------------------------------------------
print("\n[6] Número de grupos únicos por clase")
groups_per_class = (
    df_worst_cases.groupby("label")["group_id"]
    .nunique()
    .sort_index()
    .to_frame("unique_groups")
)

display(groups_per_class)

min_groups_in_class = groups_per_class["unique_groups"].min()
print(f"Mínimo número de grupos en una clase: {min_groups_in_class}")

# -------------------------------------------------------------------
# 7) Viabilidad para StratifiedGroupKFold con n_splits=5
# -------------------------------------------------------------------
N_SPLITS = 5

print(f"\n[7] Viabilidad para StratifiedGroupKFold con n_splits={N_SPLITS}")

if min_groups_in_class < N_SPLITS:
    print(f"❌ No es viable usar {N_SPLITS} folds.")
    print("   Al menos una clase tiene menos grupos únicos que el número de folds.")
else:
    print(f"✅ En principio sí es viable usar {N_SPLITS} folds.")
    print("   Cada clase tiene al menos tantos grupos únicos como folds.")

# -------------------------------------------------------------------
# 8) Tabla cruzada clase vs grupo (a nivel grupos)
#    Cuenta cuántos grupos hay de cada clase
# -------------------------------------------------------------------
print("\n[8] Tabla de grupos por clase")
group_class_df = (
    df_worst_cases.groupby("group_id")["label"]
    .first()
    .reset_index()
)

group_class_counts = group_class_df["label"].value_counts().sort_index().to_frame("n_groups")
display(group_class_counts)

# -------------------------------------------------------------------
# 9) Revisión de nulos / duplicados básicos
# -------------------------------------------------------------------
print("\n[9] Validación de nulos y duplicados")

nulls_summary = df_worst_cases[["image_path", "label", "label_id", "group_id"]].isnull().sum()
print("Nulos por columna:")
print(nulls_summary)

duplicated_paths = df_worst_cases["image_path"].duplicated().sum()
print(f"\nRutas de imagen duplicadas: {duplicated_paths}")

if duplicated_paths > 0:
    print("⚠️ Hay rutas duplicadas. Ejemplos:")
    display(df_worst_cases[df_worst_cases["image_path"].duplicated(keep=False)].head(20))
else:
    print("✅ No hay rutas duplicadas.")

# -------------------------------------------------------------------
# 10) Resumen final
# -------------------------------------------------------------------
print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)
print(f"Total de imágenes: {len(df_worst_cases)}")
print(f"Total de grupos únicos: {n_unique_groups}")
print(f"Clases: {sorted(df_worst_cases['label'].unique().tolist())}")
print(f"Mínimo de grupos en una clase: {min_groups_in_class}")
print(f"n_splits propuesto: {N_SPLITS}")

if len(bad_groups) == 0 and min_groups_in_class >= N_SPLITS:
    print("✅ Dataset listo para intentar StratifiedGroupKFold.")
else:
    print("⚠️ Hay aspectos a revisar antes de aplicar StratifiedGroupKFold.")

VALIDACIÓN PREVIA A STRATIFIEDGROUPKFOLD

[1] Tamaños de X, y, groups
len(X):      12649
len(y):      12649
len(groups): 12649
✅ Longitudes consistentes.

[2] Tamaño del dataframe final
Filas en df_worst_cases: 12649
✅ df_worst_cases coincide con X.

[3] Distribución de clases (imágenes)


,count,percent
label,,
CNV,3240,25.61
DRUSEN,3742,29.58
NORMAL,5667,44.80



Distribución por label_id:
label_id
0    3240
1    3742
2    5667
Name: count, dtype: int64

[4] Información de grupos
Número de grupos únicos: 441

Resumen de imágenes por grupo:
count    441.000000
mean      28.682540
std       17.534353
min        1.000000
25%       16.000000
50%       25.000000
75%       31.000000
max       62.000000
dtype: float64

Top 10 grupos con más imágenes:


,n_images
group_id,
NORMAL_82,62
NORMAL_83,62
NORMAL_98,62
NORMAL_84,62
DRUSEN_64,62
NORMAL_69,62
DRUSEN_86,62
NORMAL_1,62
NORMAL_101,62



Top 10 grupos con menos imágenes:


...

Mínimo número de grupos en una clase: 120

[7] Viabilidad para StratifiedGroupKFold con n_splits=5
✅ En principio sí es viable usar 5 folds.
   Cada clase tiene al menos tantos grupos únicos como folds.

[8] Tabla de grupos por clase


,n_groups
label,
CNV,161
DRUSEN,160
NORMAL,120



[9] Validación de nulos y duplicados
Nulos por columna:
image_path    0
label         0
label_id      0
group_id      0
dtype: int64

Rutas de imagen duplicadas: 8
⚠️ Hay rutas duplicadas. Ejemplos:


,patient_id,patient_class,eye,bscan,label,relative_path,image_path,exists,group_id,label_id
2324,58,CNV,OS,14,CNV,CNV/58/014_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2325,58,CNV,OS,14,CNV,CNV/58/014_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2326,58,CNV,OS,15,CNV,CNV/58/015_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2327,58,CNV,OS,15,CNV,CNV/58/015_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2328,58,CNV,OS,16,CNV,CNV/58/016_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2329,58,CNV,OS,16,CNV,CNV/58/016_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2330,58,CNV,OS,17,CNV,CNV/58/017_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2331,58,CNV,OS,17,CNV,CNV/58/017_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2332,58,CNV,OS,18,CNV,CNV/58/018_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0
2333,58,CNV,OS,18,CNV,CNV/58/018_CNV.tif,/content/drive/MyDrive/TesisIA/NEH_UT_2021Reti...,True,CNV_58,0



RESUMEN FINAL
Total de imágenes: 12649
Total de grupos únicos: 441
Clases: ['CNV', 'DRUSEN', 'NORMAL']
Mínimo de grupos en una clase: 120
n_splits propuesto: 5
✅ Dataset listo para intentar StratifiedGroupKFold.


In [ ]:
from sklearn.model_selection import train_test_split

# Trabajamos sobre el dataset final ya filtrado y con rutas válidas
df_split = df_worst_cases.copy().reset_index(drop=True)

# Un registro por grupo con su etiqueta
group_df = (
    df_split.groupby("group_id", as_index=False)
    .agg(label_id=("label_id", "first"),
         label=("label", "first"))
)

print("📌 Resumen de grupos antes del split:")
print(f"Total de imágenes: {len(df_split)}")
print(f"Total de grupos: {len(group_df)}")

print("\nDistribución de grupos por clase:")
print(group_df["label"].value_counts())

# Split estratificado a nivel grupo para reservar test fijo
train_group_ids, test_group_ids = train_test_split(
    group_df["group_id"].values,
    test_size=0.2,
    random_state=42,
    stratify=group_df["label_id"].values
)

# Construir dataframes finales
df_train = df_split[df_split["group_id"].isin(train_group_ids)].copy().reset_index(drop=True)
df_test  = df_split[df_split["group_id"].isin(test_group_ids)].copy().reset_index(drop=True)

print("\n📊 Split inicial (hold-out test):")
print(f"Train imágenes: {len(df_train)}")
print(f"Test imágenes:  {len(df_test)}")

print("\n📊 Distribución TRAIN (imágenes):")
print(df_train["label"].value_counts())

print("\n📊 Distribución TEST (imágenes):")
print(df_test["label"].value_counts())

print("\n👥 Grupos:")
print(f"Train grupos: {df_train['group_id'].nunique()}")
print(f"Test grupos:  {df_test['group_id'].nunique()}")

# Verificación de fuga
shared_groups = set(df_train["group_id"]).intersection(set(df_test["group_id"]))
print(f"\n🔍 Grupos compartidos entre TRAIN y TEST: {len(shared_groups)}")
assert len(shared_groups) == 0, "❌ Hay fuga de grupos entre train y test."
print("✅ No hay fuga entre train y test.")

📌 Resumen de grupos antes del split:
Total de imágenes: 12649
Total de grupos: 441

Distribución de grupos por clase:
label
CNV       161
DRUSEN    160
NORMAL    120
Name: count, dtype: int64

📊 Split inicial (hold-out test):
Train imágenes: 10128
Test imágenes:  2521

📊 Distribución TRAIN (imágenes):
label
NORMAL    4585
DRUSEN    2903
CNV       2640
Name: count, dtype: int64

📊 Distribución TEST (imágenes):
label
NORMAL    1082
DRUSEN     839
CNV        600
Name: count, dtype: int64

👥 Grupos:
Train grupos: 352
Test grupos:  89

🔍 Grupos compartidos entre TRAIN y TEST: 0
✅ No hay fuga entre train y test.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Base para cross-validation: solo el conjunto de entrenamiento
df_cv = df_train.copy().reset_index(drop=True)

X_cv = df_cv["image_path"].values
y_cv = df_cv["label_id"].values
groups_cv = df_cv["group_id"].values

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_summaries = []
fold_data = []

print("=" * 90)
print("GENERACIÓN DE FOLDS CON STRATIFIEDGROUPKFOLD")
print("=" * 90)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_cv, y_cv, groups_cv), start=1):
    df_fold_train = df_cv.iloc[train_idx].copy().reset_index(drop=True)
    df_fold_val   = df_cv.iloc[val_idx].copy().reset_index(drop=True)

    # Verificación de fuga
    overlap = set(df_fold_train["group_id"]).intersection(set(df_fold_val["group_id"]))
    assert len(overlap) == 0, f"❌ Fuga de grupos en fold {fold}"

    train_class_counts = df_fold_train["label"].value_counts().sort_index()
    val_class_counts   = df_fold_val["label"].value_counts().sort_index()

    summary = {
        "fold": fold,
        "train_images": len(df_fold_train),
        "val_images": len(df_fold_val),
        "train_groups": df_fold_train["group_id"].nunique(),
        "val_groups": df_fold_val["group_id"].nunique(),
    }

    for cls in sorted(df_cv["label"].unique()):
        summary[f"train_{cls}"] = int(train_class_counts.get(cls, 0))
        summary[f"val_{cls}"]   = int(val_class_counts.get(cls, 0))

    fold_summaries.append(summary)

    fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "df_train_fold": df_fold_train,
        "df_val_fold": df_fold_val
    })

    print(f"\nFold {fold}")
    print(f"  Train imágenes: {len(df_fold_train)}")
    print(f"  Val imágenes:   {len(df_fold_val)}")
    print(f"  Train grupos:   {df_fold_train['group_id'].nunique()}")
    print(f"  Val grupos:     {df_fold_val['group_id'].nunique()}")
    print("  Distribución train:")
    print(train_class_counts.to_dict())
    print("  Distribución val:")
    print(val_class_counts.to_dict())

fold_summary_df = pd.DataFrame(fold_summaries)
print("\n" + "=" * 90)
print("RESUMEN DE FOLDS")
print("=" * 90)
display(fold_summary_df)

GENERACIÓN DE FOLDS CON STRATIFIEDGROUPKFOLD

Fold 1
  Train imágenes: 8190
  Val imágenes:   1938
  Train grupos:   283
  Val grupos:     69
  Distribución train:
{'CNV': 2123, 'DRUSEN': 2464, 'NORMAL': 3603}
  Distribución val:
{'CNV': 517, 'DRUSEN': 439, 'NORMAL': 982}

Fold 2
  Train imágenes: 8214
  Val imágenes:   1914
  Train grupos:   282
  Val grupos:     70
  Distribución train:
{'CNV': 2052, 'DRUSEN': 2467, 'NORMAL': 3695}
  Distribución val:
{'CNV': 588, 'DRUSEN': 436, 'NORMAL': 890}

Fold 3
  Train imágenes: 7876
  Val imágenes:   2252
  Train grupos:   282
  Val grupos:     70
  Distribución train:
{'CNV': 2130, 'DRUSEN': 2166, 'NORMAL': 3580}
  Distribución val:
{'CNV': 510, 'DRUSEN': 737, 'NORMAL': 1005}

Fold 4
  Train imágenes: 8195
  Val imágenes:   1933
  Train grupos:   280
  Val grupos:     72
  Distribución train:
{'CNV': 2200, 'DRUSEN': 2219, 'NORMAL': 3776}
  Distribución val:
{'CNV': 440, 'DRUSEN': 684, 'NORMAL': 809}

Fold 5
  Train imágenes: 8037
  Val imáge

,fold,train_images,val_images,train_groups,val_groups,train_CNV,val_CNV,train_DRUSEN,val_DRUSEN,train_NORMAL,val_NORMAL
0,1,8190,1938,283,69,2123,517,2464,439,3603,982
1,2,8214,1914,282,70,2052,588,2467,436,3695,890
2,3,7876,2252,282,70,2130,510,2166,737,3580,1005
3,4,8195,1933,280,72,2200,440,2219,684,3776,809
4,5,8037,2091,281,71,2055,585,2296,607,3686,899


In [ ]:
import os
import pandas as pd

# ============================================================
# CONFIGURACIÓN DE SALIDA
# ============================================================
OUTPUT_DIR = "/content/drive/MyDrive/TesisIA/splits_cv_5fold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 90)
print("GUARDANDO CSVs MAESTROS Y ASIGNACIONES DE SPLITS/FOLDS")
print("=" * 90)

# ============================================================
# 1) CSV maestro limpio final
# ============================================================
master_clean_path = os.path.join(OUTPUT_DIR, "dataset_master_clean.csv")
df_worst_cases.to_csv(master_clean_path, index=False)

print(f"✅ Guardado: {master_clean_path}")
print(f"   Filas: {len(df_worst_cases)}")

# ============================================================
# 2) Split externo train / test
# ============================================================
train_outer_path = os.path.join(OUTPUT_DIR, "train_outer_split.csv")
test_outer_path  = os.path.join(OUTPUT_DIR, "test_outer_split.csv")

df_train.to_csv(train_outer_path, index=False)
df_test.to_csv(test_outer_path, index=False)

print(f"✅ Guardado: {train_outer_path}")
print(f"   Filas: {len(df_train)}")

print(f"✅ Guardado: {test_outer_path}")
print(f"   Filas: {len(df_test)}")

# ============================================================
# 3) Resumen tabular de folds
# ============================================================
fold_summary_path = os.path.join(OUTPUT_DIR, "fold_summary.csv")
fold_summary_df.to_csv(fold_summary_path, index=False)

print(f"✅ Guardado: {fold_summary_path}")
print(f"   Filas: {len(fold_summary_df)}")

# ============================================================
# 4) Guardar CSVs individuales por fold
# ============================================================
for item in fold_data:
    fold = item["fold"]
    df_train_fold = item["df_train_fold"]
    df_val_fold   = item["df_val_fold"]

    train_fold_path = os.path.join(OUTPUT_DIR, f"fold_{fold}_train.csv")
    val_fold_path   = os.path.join(OUTPUT_DIR, f"fold_{fold}_val.csv")

    df_train_fold.to_csv(train_fold_path, index=False)
    df_val_fold.to_csv(val_fold_path, index=False)

    print(f"✅ Guardado: {train_fold_path} ({len(df_train_fold)} filas)")
    print(f"✅ Guardado: {val_fold_path} ({len(df_val_fold)} filas)")

# ============================================================
# 5) CSV largo con todas las asignaciones de folds
#    (recomendado para reproducibilidad y trazabilidad)
# ============================================================
records = []

# ---- Test externo
for _, row in df_test.iterrows():
    records.append({
        "image_path": row["image_path"],
        "relative_path": row.get("relative_path", None),
        "label": row["label"],
        "label_id": row["label_id"],
        "group_id": row["group_id"],
        "outer_split": "test",
        "fold": -1,
        "role": "test"
    })

# ---- Folds internos del train externo
for item in fold_data:
    fold = item["fold"]

    for _, row in item["df_train_fold"].iterrows():
        records.append({
            "image_path": row["image_path"],
            "relative_path": row.get("relative_path", None),
            "label": row["label"],
            "label_id": row["label_id"],
            "group_id": row["group_id"],
            "outer_split": "train",
            "fold": fold,
            "role": "train"
        })

    for _, row in item["df_val_fold"].iterrows():
        records.append({
            "image_path": row["image_path"],
            "relative_path": row.get("relative_path", None),
            "label": row["label"],
            "label_id": row["label_id"],
            "group_id": row["group_id"],
            "outer_split": "train",
            "fold": fold,
            "role": "val"
        })

df_folds_long = pd.DataFrame(records)

fold_assignments_long_path = os.path.join(OUTPUT_DIR, "fold_assignments_long.csv")
df_folds_long.to_csv(fold_assignments_long_path, index=False)

print(f"✅ Guardado: {fold_assignments_long_path}")
print(f"   Filas: {len(df_folds_long)}")

# ============================================================
# 6) CSV compacto de grupos (útil para auditoría)
# ============================================================
group_records = []

# grupos del test externo
for gid in sorted(df_test["group_id"].unique()):
    temp = df_test[df_test["group_id"] == gid]
    group_records.append({
        "group_id": gid,
        "label": temp["label"].iloc[0],
        "label_id": temp["label_id"].iloc[0],
        "outer_split": "test",
        "fold": -1,
        "role": "test",
        "n_images": len(temp)
    })

# grupos por fold
for item in fold_data:
    fold = item["fold"]

    # train groups
    train_group_sizes = item["df_train_fold"].groupby("group_id").size()
    train_group_meta = item["df_train_fold"].groupby("group_id")[["label", "label_id"]].first()

    for gid, n_imgs in train_group_sizes.items():
        group_records.append({
            "group_id": gid,
            "label": train_group_meta.loc[gid, "label"],
            "label_id": train_group_meta.loc[gid, "label_id"],
            "outer_split": "train",
            "fold": fold,
            "role": "train",
            "n_images": int(n_imgs)
        })

    # val groups
    val_group_sizes = item["df_val_fold"].groupby("group_id").size()
    val_group_meta = item["df_val_fold"].groupby("group_id")[["label", "label_id"]].first()

    for gid, n_imgs in val_group_sizes.items():
        group_records.append({
            "group_id": gid,
            "label": val_group_meta.loc[gid, "label"],
            "label_id": val_group_meta.loc[gid, "label_id"],
            "outer_split": "train",
            "fold": fold,
            "role": "val",
            "n_images": int(n_imgs)
        })

df_group_assignments = pd.DataFrame(group_records)

group_assignments_path = os.path.join(OUTPUT_DIR, "group_assignments_long.csv")
df_group_assignments.to_csv(group_assignments_path, index=False)

print(f"✅ Guardado: {group_assignments_path}")
print(f"   Filas: {len(df_group_assignments)}")

# ============================================================
# 7) Archivo README-resumen en CSV simple
# ============================================================
summary_info = pd.DataFrame([
    {"file": "dataset_master_clean.csv", "description": "Dataset final limpio tras filtrado y verificación de rutas", "rows": len(df_worst_cases)},
    {"file": "train_outer_split.csv", "description": "Conjunto de entrenamiento externo (80%)", "rows": len(df_train)},
    {"file": "test_outer_split.csv", "description": "Conjunto de prueba externo (20%)", "rows": len(df_test)},
    {"file": "fold_summary.csv", "description": "Resumen tabular de imágenes y grupos por fold", "rows": len(fold_summary_df)},
    {"file": "fold_assignments_long.csv", "description": "Asignación larga por imagen a test/train/val en cada fold", "rows": len(df_folds_long)},
    {"file": "group_assignments_long.csv", "description": "Asignación larga por grupo/paciente a test/train/val en cada fold", "rows": len(df_group_assignments)},
])

readme_csv_path = os.path.join(OUTPUT_DIR, "files_manifest.csv")
summary_info.to_csv(readme_csv_path, index=False)

print(f"✅ Guardado: {readme_csv_path}")

print("\n" + "=" * 90)
print("PROCESO FINALIZADO")
print("=" * 90)
print(f"Todos los archivos se guardaron en: {OUTPUT_DIR}")

print("\nArchivos principales recomendados para usar después:")
print("- dataset_master_clean.csv")
print("- train_outer_split.csv")
print("- test_outer_split.csv")
print("- fold_summary.csv")
print("- fold_assignments_long.csv")
print("- group_assignments_long.csv")

GUARDANDO CSVs MAESTROS Y ASIGNACIONES DE SPLITS/FOLDS
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/dataset_master_clean.csv
   Filas: 12649
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/train_outer_split.csv
   Filas: 10128
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/test_outer_split.csv
   Filas: 2521
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_summary.csv
   Filas: 5
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_1_train.csv (8190 filas)
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_1_val.csv (1938 filas)
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_2_train.csv (8214 filas)
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_2_val.csv (1914 filas)
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_3_train.csv (7876 filas)
✅ Guardado: /content/drive/MyDrive/TesisIA/splits_cv_5fold/fold_3_val.csv (2252 filas)
✅ Guardado: /content/drive/MyDrive/Te

In [ ]:
print("dataset_master_clean:", pd.read_csv("splits_cv_5fold/dataset_master_clean.csv").shape)
print("train_outer_split:", pd.read_csv("splits_cv_5fold/train_outer_split.csv").shape)
print("test_outer_split:", pd.read_csv("splits_cv_5fold/test_outer_split.csv").shape)
print("fold_summary:", pd.read_csv("splits_cv_5fold/fold_summary.csv").shape)
print("fold_assignments_long:", pd.read_csv("splits_cv_5fold/fold_assignments_long.csv").shape)
print("group_assignments_long:", pd.read_csv("splits_cv_5fold/group_assignments_long.csv").shape)

dataset_master_clean: (12649, 10)
train_outer_split: (10128, 10)
test_outer_split: (2521, 10)
fold_summary: (5, 11)
fold_assignments_long: (53161, 8)
group_assignments_long: (1849, 7)
